# visualization_010

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (82).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 12 |
| Detected functions | plot_trade_summary, assign_time_slot, assign_time_slot |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/tradebook.csv')
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the datetime column is in datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'], format='%Y-%m-%d %H:%M:%S')

# 1. Line Plot for PnL over Time
plt.figure(figsize=(10, 6))
plt.plot(df['Entry Time'], df['PnL'], label='PnL Over Time', color='blue')
plt.xlabel('Entry Time')
plt.ylabel('PnL')
plt.title('PnL Over Time')
plt.legend()
plt.grid()
plt.show()

# 2. Scatter Plot for Entry Price vs Exit Price
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df['Entry Price'], y=df['Exit Price'], hue=df['Ticker'])
plt.title('Entry Price vs Exit Price')
plt.xlabel('Entry Price')
plt.ylabel('Exit Price')
plt.grid()
plt.show()

# 3. Bar Plot for PnL by Action

plt.figure(figsize=(10, 6))
df.groupby('Exit Reason')['PnL'].mean().plot(kind='bar', color='orange')
plt.title('Average PnL by Action')
plt.xlabel('Action')
plt.ylabel('Average PnL')
plt.grid(axis='y')
plt.show()

# 4. Histogram for PnL Distribution
plt.figure(figsize=(8, 6))
sns.histplot(df['PnL'], kde=True, bins=50, color='green')
plt.title('PnL Distribution')
plt.xlabel('PnL')
plt.ylabel('Frequency')
plt.grid()
plt.show()

# 5. Line Plot for Balance over Time
plt.figure(figsize=(10, 6))
plt.plot(df['Entry Time'], df['Cumulative PnL'], label='Balance Over Time', color='purple')
plt.xlabel('Entry Time')
plt.ylabel('Cumulative PnL')
plt.title('Balance Over Time')
plt.legend()
plt.grid()
plt.show()

# 6. Box Plot for PnL by Ticker
plt.figure(figsize=(12, 6))
sns.boxplot(x='Ticker', y='PnL', data=df)
plt.title('PnL by Ticker')
plt.xlabel('Ticker')
plt.ylabel('PnL')
plt.xticks(rotation=45)
plt.grid()
plt.show()

# 7. Heatmap for Correlation
plt.figure(figsize=(10, 6))
correlation = df[['Entry Price', 'Exit Price', 'PnL', 'Cumulative PnL']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# 8. KDE Plot for PnL
plt.figure(figsize=(8, 6))
sns.kdeplot(df['PnL'], shade=True, color='red')
plt.title('KDE Plot for PnL')
plt.xlabel('PnL')
plt.ylabel('Density')
plt.grid()
plt.show()

# 9. Pair Plot for Numerical Columns
numerical_columns = ['Entry Price', 'Exit Price', 'PnL', 'Cumulative PnL']
sns.pairplot(df[numerical_columns])
plt.show()

# 10. Time Series of PnL with Moving Average
plt.figure(figsize=(10, 6))
df['PnL_MA'] = df['PnL'].rolling(window=50).mean()
plt.plot(df['Entry Time'], df['PnL'], label='PnL', alpha=0.5)
plt.plot(df['Entry Time'], df['PnL_MA'], label='50-Point Moving Average', color='red')
plt.xlabel('Entry Time')
plt.ylabel('PnL')
plt.title('PnL with Moving Average')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import plotly.express as px

def plot_trade_summary(tradebook_df):
    # Determine winning and losing trades
    tradebook_df["Win"] = tradebook_df["PnL"] > 0
    tradebook_df["Loss"] = tradebook_df["PnL"] < 0

    # Group by Ticker and calculate summary stats
    summary = tradebook_df.groupby("Ticker").agg(
        Total_Trades=("Ticker", "count"),
        Winning_Trades=("Win", "sum"),
        Losing_Trades=("Loss", "sum")
    ).reset_index()

    # Melt the data for grouped bar chart
    summary_melted = summary.melt(id_vars="Ticker",
                                  value_vars=["Winning_Trades", "Losing_Trades"],
                                  var_name="Trade Type",
                                  value_name="Count")

    # Create Plotly bar chart
    fig = px.bar(summary_melted,
                 x="Ticker",
                 y="Count",
                 color="Trade Type",
                 barmode="group",
                 title="Trade Summary by Ticker",
                 labels={"Count": "Number of Trades", "Ticker": "Stock Ticker"},
                 text_auto=True)

    fig.update_layout(xaxis_tickangle=-45)  # Rotate x-axis labels for better readability
    fig.show()

# Plot trade summary
plot_trade_summary(df)


In [ ]:
# 6. Bar Plot for Sum of PnL by Ticker
PnL_by_ticker = df.groupby('Ticker')['PnL'].sum()

plt.figure(figsize=(20, 7))
PnL_by_ticker.plot(kind='bar', color='purple')
plt.title('Sum of PnL by Ticker')
plt.xlabel('Ticker')
plt.ylabel('Sum of PnL')
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.show()


In [ ]:
import plotly.express as px

# Ensure 'Exit Time' or 'Exit Date' is in datetime format
df['Exit Time'] = pd.to_datetime(df['Exit Time'])

# Extract the weekday (0 = Monday, 1 = Tuesday, ..., 6 = Sunday)
df['Weekday'] = df['Exit Time'].dt.day_name()

# Group by Weekday and calculate the sum of PnL for each day
weekday_PnL = df.groupby('Weekday')['PnL'].sum().reset_index()

# Reorder the weekdays to have a correct chronological order
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
weekday_PnL['Weekday'] = pd.Categorical(weekday_PnL['Weekday'], categories=weekday_order, ordered=True)
weekday_PnL = weekday_PnL.sort_values('Weekday')

# Plot the bar chart
fig = px.bar(weekday_PnL, x='Weekday', y='PnL', title="PnL by Weekday",
             labels={'PnL': 'Total PnL', 'Weekday': 'Weekday'},
             color='PnL', color_continuous_scale='Viridis')

# Show the plot
fig.show()


In [ ]:
# 5. Line Plot for Balance over Time
plt.figure(figsize=(10, 6))
plt.plot(df['Entry Time'], df['Cumulative PnL'], label='Balance Over Time', color='purple')
plt.xlabel('DateTime')
plt.ylabel('Balance')
plt.title('Balance Over Time')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

# Load the tradebook DataFrame
tradebook = df

# Convert 'Entry Time' to datetime if not already
tradebook['Entry Time'] = pd.to_datetime(tradebook['Entry Time'])

# Define the start and end time for the trading session
start_time = pd.Timestamp("09:15:00")
end_time = pd.Timestamp("15:30:00")

# Create a function to assign time slots
def assign_time_slot(entry_time):
    minutes_since_start = (entry_time - start_time).seconds // 60
    slot_number = minutes_since_start // 10
    slot_start_time = start_time + pd.Timedelta(minutes=slot_number * 10)
    slot_end_time = slot_start_time + pd.Timedelta(minutes=10)
    return f"{slot_start_time.time()} - {slot_end_time.time()}"

# Apply the function to create time slots
tradebook['Time Slot'] = tradebook['Entry Time'].apply(assign_time_slot)

# Create a new column to categorize PnL as positive or negative
tradebook['PnL Category'] = tradebook['PnL'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

# Define a color map for positive and negative PnL
color_map = {
    'Positive': 'green',
    'Negative': 'red'
}

# Create a scatter plot using Plotly
fig = px.scatter(
    tradebook,
    x='Time Slot',  # Use the 'Time Slot' column for x-axis
    y='PnL',
    title='PnL vs Time Slot',
    labels={'Time Slot': '10-Minute Time Slot', 'PnL': 'PnL'},
    color='PnL Category',  # Use the category for color
    color_discrete_map=color_map
)

# Customize the layout
fig.update_layout(
    xaxis_title="10-Minute Time Slot",
    yaxis_title="PnL",
    template="plotly_dark",
    xaxis=dict(categoryorder='array', categoryarray=sorted(tradebook['Time Slot'].unique()))
)

# Show the plot
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px

# Load the tradebook DataFrame
tradebook = df

# Convert 'Entry Time' to datetime if not already
tradebook['Entry Time'] = pd.to_datetime(tradebook['Exit Time'])

# Define the start and end time for the trading session
start_time = pd.Timestamp("09:15:00")
end_time = pd.Timestamp("15:30:00")

# Create a function to assign time slots
def assign_time_slot(entry_time):
    minutes_since_start = (entry_time - start_time).seconds // 60
    slot_number = minutes_since_start // 10
    slot_start_time = start_time + pd.Timedelta(minutes=slot_number * 10)
    slot_end_time = slot_start_time + pd.Timedelta(minutes=10)
    return f"{slot_start_time.time()} - {slot_end_time.time()}"

# Apply the function to create time slots
tradebook['Time Slot'] = tradebook['Entry Time'].apply(assign_time_slot)

# Create a new column to categorize PnL as positive or negative
tradebook['PnL Category'] = tradebook['PnL'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

# Define a color map for positive and negative PnL
color_map = {
    'Positive': 'green',
    'Negative': 'red'
}

# Create a scatter plot using Plotly
fig = px.scatter(
    tradebook,
    x='Time Slot',  # Use the 'Time Slot' column for x-axis
    y='PnL',
    title='PnL vs Time Slot',
    labels={'Time Slot': '10-Minute Time Slot', 'PnL': 'PnL'},
    color='PnL Category',  # Use the category for color
    color_discrete_map=color_map
)

# Customize the layout
fig.update_layout(
    xaxis_title="10-Minute Time Slot",
    yaxis_title="PnL",
    template="plotly_dark",
    xaxis=dict(categoryorder='array', categoryarray=sorted(tradebook['Time Slot'].unique()))
)

# Show the plot
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import calendar

# Assuming you have a DataFrame named tradebook
# tradebook contains columns: 'Entry Time' and 'PnL'

# Ensure 'Entry Time' is in datetime format
tradebook['Entry Time'] = pd.to_datetime(tradebook['Entry Time'])

# Extract month and year from 'Entry Time'
tradebook['Year'] = tradebook['Entry Time'].dt.year
tradebook['Month'] = tradebook['Entry Time'].dt.month

# Calculate monthly total profit percent
monthly_profit = tradebook.groupby(['Year', 'Month'])['PnL'].sum().reset_index()
monthly_profit['Profit Percent'] = (monthly_profit['PnL'] / 5000000) * 100

# Calculate yearly return
yearly_return = tradebook.groupby('Year')['PnL'].sum().reset_index()
yearly_return['Profit Percent'] = (yearly_return['PnL'] / 5000000) * 100

# Create a pivot table for heatmap (Years as rows, Months as columns)
pivot_table = monthly_profit.pivot(index='Year', columns='Month', values='Profit Percent')

# Convert months to abbreviations for labels
pivot_table.columns = [calendar.month_abbr[i] for i in pivot_table.columns]

# Plot heatmap using Plotly
heatmap = px.imshow(
    pivot_table,
    text_auto=".2f",
    color_continuous_scale="YlGnBu",
    labels={"color": "Profit %"},
    title="Monthly Total Profit Percentage",
)

heatmap.update_layout(
    xaxis_title="Month",
    yaxis_title="Year",
    xaxis=dict(tickmode="array", tickvals=list(range(12)), ticktext=list(pivot_table.columns)),
    yaxis=dict(tickmode="array", tickvals=pivot_table.index),
    coloraxis_colorbar=dict(title="Profit %"),
)

heatmap.show()

# Plot yearly return using Plotly
bar_chart = go.Figure()

bar_chart.add_trace(go.Bar(
    x=yearly_return['Year'],
    y=yearly_return['Profit Percent'],
    marker=dict(color="skyblue"),
    text=yearly_return['Profit Percent'].round(2),
    textposition='outside',
))

bar_chart.update_layout(
    title="Yearly Total Profit Percentage",
    xaxis_title="Year",
    yaxis_title="Yearly Return (%)",
    bargap=0.2,
    template="plotly_white"
)

bar_chart.show()


In [ ]:
!pip install PdfPages

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os

# Define the output PDF path
pdf_path = "/content/drive/MyDrive/tradebook_plots.pdf"

# Create a multi-page PDF
with PdfPages(pdf_path) as pdf:
    for i in plt.get_fignums():  # Get all active figures
        fig = plt.figure(i)
        pdf.savefig(fig)  # Save figure
        plt.close(fig)  # Close to free memory

print(f"All figures saved in: {pdf_path}")